# Walmart Store Sales Forecasting: Model Inference

This notebook fulfils the assignment's inference requirements:

- The team's **best model** (by shared validation WMAE) is **registered in the MLflow Model Registry**.
- The model is **loaded directly from the registry** (not from a local file, not retrained).
- It predicts on the **raw, unpreprocessed test set** and generates the final submission file.

**Final model comparison on the shared validation split** (39-week window, Nov 2011 - Jul 2012, chosen to match the test period's holiday composition; metric: WMAE, holiday weeks weighted 5x; lower is better):

| Model | Architecture family | Best val WMAE | Notebook |
|---|---|---|---|
| **LightGBM** | gradient-boosted trees | **1811** | model_experiment_LightGBM |
| XGBoost | gradient-boosted trees | 1839 | model_experiment_XGBoost |
| N-BEATS | deep learning | 2342 | model_experiment_NBEATS |
| PatchTST | deep learning (Transformer) | 2529 | model_experiment_PatchTST |
| DLinear | deep learning (linear) | 3386 | model_experiment_DLinear |
| seasonal naive (benchmark) | - | ~2045 | inside LightGBM notebook (blend alpha=1.0) |

**Winner: LightGBM** (direct multi-horizon + seasonal-naive holiday blend). The ranking matches published competition evidence (M4 / M5): on short, feature-rich retail series, gradient-boosted trees with engineered features beat neural forecasters.

Notes for honest reporting: the DL models score the 36 cold-start test rows via the shared fallback while the tree evaluation dropped 413 cold validation rows (small asymmetry, documented in the README); the Kaggle competition no longer accepts submissions, so models are compared on the holdout described above.

## Setup

Kaggle settings: Internet ON, competition dataset attached, `DAGSHUB_TOKEN` secret enabled. The winning pipeline was trained with LightGBM, so its library must be present for unpickling.

In [ ]:
!pip install -q mlflow dagshub lightgbm

In [ ]:
import os
import numpy as np
import pandas as pd
import mlflow

import dagshub
from kaggle_secrets import UserSecretsClient

dagshub.auth.add_app_token(UserSecretsClient().get_secret('DAGSHUB_TOKEN'))
dagshub.init(repo_owner='Nestor-Dzadzamia', repo_name='walmart-sales-forecasting', mlflow=True)
print('tracking uri:', mlflow.get_tracking_uri())
print('mlflow', mlflow.__version__)

## Locate the winning pipeline run

The cell below lists the candidate runs from the winning architecture's experiment so the exact run holding the pipeline artifact can be identified. The constants cell after it pins the choice explicitly (reproducibility: this notebook should not silently follow whatever run happens to sort first).

In [ ]:
candidates = mlflow.search_runs(
    experiment_names=['LightGBM_Training'],
    output_format='pandas',
)
cols = [c for c in ['run_id', 'tags.mlflow.runName', 'metrics.validation_wmae',
                    'metrics.val_wmae', 'end_time'] if c in candidates.columns]
display(candidates[cols].sort_values('end_time', ascending=False).head(15))

In [ ]:
# Pin the winning run explicitly (from the table above / DagsHub UI):
# the LightGBM run that logged the final pyfunc pipeline artifact.
BEST_RUN_ID = 'PASTE_RUN_ID_HERE'
ARTIFACT_PATH = 'pipeline'   # the artifact name used in mlflow.pyfunc.log_model(...)
REGISTRY_NAME = 'walmart-weekly-sales-best-model'

assert BEST_RUN_ID != 'PASTE_RUN_ID_HERE', 'paste the winning run id first'
best_model_uri = f'runs:/{BEST_RUN_ID}/{ARTIFACT_PATH}'
print('registering from:', best_model_uri)

## Register the model in the Model Registry

In [ ]:
registered = mlflow.register_model(model_uri=best_model_uri, name=REGISTRY_NAME)
print('registered:', registered.name, '| version:', registered.version)

## Load the model FROM THE REGISTRY

This is the assignment's key requirement: the model comes out of the registry by name and version, with no reference to local files or training code.

In [ ]:
registry_uri = f'models:/{REGISTRY_NAME}/{registered.version}'
print('loading:', registry_uri)
model = mlflow.pyfunc.load_model(registry_uri)
print('loaded:', type(model).__name__)

## Predict on the raw test set

The pipeline contract (identical across all five model notebooks): raw `test.csv` frame in, final `Weekly_Sales` predictions out. Merging, preprocessing, forecasting, cold-start fallback and clipping all happen inside the registered pipeline.

In [ ]:
path = '/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/'
test = pd.read_csv(path + 'test.csv.zip')
print('raw test:', test.shape)

predictions = model.predict(test)
print('predictions:', predictions.shape)
predictions.head()

## Sanity checks and final submission

In [ ]:
assert len(predictions) == len(test), f'row mismatch: {len(predictions)} vs {len(test)}'
assert predictions['Weekly_Sales'].notna().all(), 'NaN predictions found'
assert (predictions['Weekly_Sales'] >= 0).all(), 'negative predictions found (contract: clip at 0)'

sub = predictions.copy()
sub['Date'] = pd.to_datetime(sub['Date'])
sub['Id'] = (sub['Store'].astype(str) + '_' + sub['Dept'].astype(str)
             + '_' + sub['Date'].dt.strftime('%Y-%m-%d'))
submission = sub[['Id', 'Weekly_Sales']].sort_values('Id').reset_index(drop=True)

sample = pd.read_csv(path + 'sampleSubmission.csv.zip')
assert len(submission) == len(sample), 'row count differs from sampleSubmission'
assert set(submission['Id']) == set(sample['Id']), 'Id set differs from sampleSubmission'

submission.to_csv('final_submission.csv', index=False)
print('final_submission.csv written:', submission.shape)
submission.head()

## Notes

- **Registered model**: the LightGBM direct multi-horizon pipeline (validation WMAE 1811), chosen over 4 alternatives on the shared holdout.
- **Why the registry**: a named, versioned model decouples inference from training notebooks; this notebook would keep working even if every training notebook changed, and a better future model becomes just a new registry version.
- **Fixed-horizon caveat** (applies to all team pipelines): each registered pipeline forecasts the specific 39-week window after its training history. The competition test set is exactly that window; other date ranges require retraining.
- The competition no longer accepts submissions, so `final_submission.csv` is the deliverable artifact and models are compared on the holdout validation described at the top.